In [2]:
pip install numpy

In [3]:
pip install pandas

In [4]:
pip install matplotlib

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [6]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200

y = np.random.choice([0, 1], size=n, p=[0.5, 0.5])

df = pd.DataFrame({
    # 1. USELESS - near constant (VarianceThreshold should catch this)
    'constant_feature': np.ones(n),  # exactly constant, variance = 0

    'almost_constant': np.random.choice([0, 1], size=n, p=[0.98, 0.02]),  # 98% zeros

    # 2. USELESS - random noise, no relation to y (SelectKBest should catch this)
    'random_noise_1': np.random.normal(0, 1, n),
    'random_noise_2': np.random.uniform(0, 100, n),

    # 3. USEFUL - strongly correlated with y (both should keep this)
    'strong_signal': y * 5 + np.random.normal(0, 1, n),  # y=1 -> higher values

    # 4. USEFUL - moderately correlated with y
    'medium_signal': y * 2 + np.random.normal(0, 3, n),

    # 5. USEFUL but binary/categorical-like, tests chi2
    'weak_binary_signal': np.where(
        y == 1,
        np.random.choice([0, 1], size=n, p=[0.3, 0.7]),
        np.random.choice([0, 1], size=n, p=[0.7, 0.3])
    ),

    # 6. Low variance but still has SOME predictive value (edge case)
    'low_var_but_useful': np.where(y == 1, 1.01, 1.00) + np.random.normal(0, 0.001, n),
})

df['target'] = y
print(df.head())
print(df.describe())

   constant_feature  almost_constant  random_noise_1  random_noise_2  \
0               1.0                0        1.305479       96.947043   
1               1.0                0        0.021004       86.550713   
2               1.0                0        0.681953       81.707207   
3               1.0                0       -0.310267       25.790283   
4               1.0                0        0.324166       17.088759   

   strong_signal  medium_signal  weak_binary_signal  low_var_but_useful  \
0       0.721672       7.580797                   0            0.998653   
1       3.870948       0.407394                   0            1.009028   
2       4.475480       0.531682                   0            1.011200   
3       5.489375       5.132483                   1            1.009343   
4      -1.222128       2.045674                   0            0.998953   

   target  
0       0  
1       1  
2       1  
3       1  
4       0  
       constant_feature  almost_constant  ra

In [7]:
from sklearn.feature_selection import VarianceThreshold , f_classif, f_regression , mutual_info_classif , mutual_info_regression

In [8]:
selector = VarianceThreshold(threshold=0) # Declaration of vairance thersold
df_reduce = selector.fit_transform(df)
print(df.columns) # original columns
#print(df_reduce)
print(df.columns[selector.get_support(indices=True)]) # columns with non zero thersold

Index(['constant_feature', 'almost_constant', 'random_noise_1',
       'random_noise_2', 'strong_signal', 'medium_signal',
       'weak_binary_signal', 'low_var_but_useful', 'target'],
      dtype='object')
Index(['almost_constant', 'random_noise_1', 'random_noise_2', 'strong_signal',
       'medium_signal', 'weak_binary_signal', 'low_var_but_useful', 'target'],
      dtype='object')


In [9]:
from sklearn.feature_selection import SelectKBest, f_classif, chi2, mutual_info_classif

In [10]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=4)
X = df.drop(columns='target')
y = df['target']
selector.fit(X, y)

scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
print(scores)

print("\nTop 4 features selected:", X.columns[selector.get_support()].tolist())

low_var_but_useful    4687.694041
strong_signal         1383.203535
weak_binary_signal      39.996549
medium_signal           18.114703
almost_constant          1.940400
random_noise_2           0.006782
random_noise_1           0.002077
constant_feature              NaN
dtype: float64

Top 4 features selected: ['strong_signal', 'medium_signal', 'weak_binary_signal', 'low_var_but_useful']


/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [11]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_regression, k=4)
X = df.drop(columns='target')
y = df['target']
selector.fit(X, y)

scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
print(scores)

print("\nTop 4 features selected:", X.columns[selector.get_support()].tolist())

low_var_but_useful    4687.694039
strong_signal         1383.203535
weak_binary_signal      39.996549
medium_signal           18.114703
almost_constant          1.940400
random_noise_2           0.006782
random_noise_1           0.002077
constant_feature         0.000000
dtype: float64

Top 4 features selected: ['strong_signal', 'medium_signal', 'weak_binary_signal', 'low_var_but_useful']


In [12]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=mutual_info_regression, k=4)
X = df.drop(columns='target')
y = df['target']
selector.fit(X, y)

scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
print(scores)

print("\nTop 4 features selected:", X.columns[selector.get_support()].tolist())

low_var_but_useful    0.695453
strong_signal         0.689870
weak_binary_signal    0.115031
almost_constant       0.063514
constant_feature      0.033007
random_noise_2        0.021341
random_noise_1        0.011269
medium_signal         0.008336
dtype: float64

Top 4 features selected: ['almost_constant', 'strong_signal', 'weak_binary_signal', 'low_var_but_useful']


In [13]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=mutual_info_classif, k=4)
X = df.drop(columns='target')
y = df['target']
selector.fit(X, y)

scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
print(scores)

print("\nTop 4 features selected:", X.columns[selector.get_support()].tolist())

low_var_but_useful    0.695453
strong_signal         0.689870
weak_binary_signal    0.051064
random_noise_2        0.021341
constant_feature      0.017225
random_noise_1        0.011269
medium_signal         0.008336
almost_constant       0.000000
dtype: float64

Top 4 features selected: ['random_noise_2', 'strong_signal', 'weak_binary_signal', 'low_var_but_useful']


In [14]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# Separate features and target
X = df.drop(columns=['target'])
y = df['target']

# Create the model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Create RFE
rfe = RFE(
    estimator=model,
    n_features_to_select=4
)

# Fit RFE
rfe.fit(X, y)

# Results
print("Selected Features:")
print(X.columns[rfe.support_].tolist())

print("\nRFE Ranking:")
for feature, rank in zip(X.columns, rfe.ranking_):
    print(f"{feature}: {rank}")

# Create dataset with selected features
X_rfe = X.loc[:, rfe.support_]

print("\nRFE Dataset:")
print(X_rfe.head())

Selected Features:
['strong_signal', 'medium_signal', 'weak_binary_signal', 'low_var_but_useful']

RFE Ranking:
constant_feature: 5
almost_constant: 4
random_noise_1: 2
random_noise_2: 3
strong_signal: 1
medium_signal: 1
weak_binary_signal: 1
low_var_but_useful: 1

RFE Dataset:
   strong_signal  medium_signal  weak_binary_signal  low_var_but_useful
0       0.721672       7.580797                   0            0.998653
1       3.870948       0.407394                   0            1.009028
2       4.475480       0.531682                   0            1.011200
3       5.489375       5.132483                   1            1.009343
4      -1.222128       2.045674                   0            0.998953


In [15]:
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

# Separate features and target
X = df.drop(columns=['target'])
y = df['target']

# Create the model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Cross-validation strategy
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Create RFECV
rfecv = RFECV(
    estimator=model,
    step=1,
    cv=cv,
    scoring='accuracy',
    min_features_to_select=1
)

# Fit RFECV
rfecv.fit(X, y)

# Results
print("Optimal number of features:")
print(rfecv.n_features_)

print("\nSelected Features:")
print(X.columns[rfecv.support_].tolist())

print("\nRFECV Ranking:")
for feature, rank in zip(X.columns, rfecv.ranking_):
    print(f"{feature}: {rank}")

# Create dataset with selected features
X_rfecv = X.loc[:, rfecv.support_]

print("\nRFECV Dataset:")
print(X_rfecv.head())

Optimal number of features:
1

Selected Features:
['low_var_but_useful']

RFECV Ranking:
constant_feature: 8
almost_constant: 7
random_noise_1: 5
random_noise_2: 6
strong_signal: 2
medium_signal: 4
weak_binary_signal: 3
low_var_but_useful: 1

RFECV Dataset:
   low_var_but_useful
0            0.998653
1            1.009028
2            1.011200
3            1.009343
4            0.998953
